## read_excel()

**Purpose:** Load data from an Excel file (.xls or .xlsx) into a pandas DataFrame.

**Why it exists:** Real-world business/ML data very often lives in Excel, not CSV — 
finance, sales, HR, government datasets are commonly shared as .xlsx files.

**Syntax:**
`pd.read_excel(path, sheet_name=0, header=0, index_col=None, names=None)`

**Return type:** DataFrame (or dict of DataFrames if sheet_name=None)

**ML Usage:** Loading raw datasets shared by teams/clients who use Excel instead of CSV/DB.


In [54]:
import pandas as pd

data = {
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'city': ['Mumbai', 'Delhi', 'Pune']
}
df_temp = pd.DataFrame(data)
df_temp.to_excel('examples/ex_excel.xlsx', sheet_name='Sheet1', index=False)

In [55]:
df = pd.read_excel('examples/ex_excel.xlsx', sheet_name='Sheet1')
df

,name,age,city
0,Alice,25,Mumbai
1,Bob,30,Delhi
2,Charlie,35,Pune


## read_json()

**Purpose:** Load data from a JSON file/string into a pandas DataFrame.

**Why it exists:** JSON is the standard format for web APIs, NoSQL databases (MongoDB), 
and config/data interchange. ML datasets pulled from APIs almost always arrive as JSON.

**Syntax:**
pd.read_json(path_or_buf, orient=None, lines=False)

**Return type:** DataFrame

**ML Usage:** Loading API responses, NoSQL exports, or nested config-like datasets.

In [56]:
import json

data = [
    {"name": "Alex", "age": 45, "city": "NY"},
    {"name": "Bean", "age": 32, "city": "Delhi"},
    {"name": "Charles", "age": 43, "city": "Mumbai"}
]

with open('examples/ex_json.json', 'w') as f:
    json.dump(data, f)

In [57]:
df=pd.read_json('examples/ex_json.json')
df

# read_json()
# Reads JSON (list of records) into a DataFrame
# Each dict in the list becomes one row
# Keys become column names automatically


,name,age,city
0,Alex,45,NY
1,Bean,32,Delhi
2,Charles,43,Mumbai


In [58]:
data.append({"name": "David", "age": 40})  # no 'city' key

with open('examples/ex_json.json', 'w') as f:
    json.dump(data, f)

pd.read_json('examples/ex_json.json')

,name,age,city
0,Alex,45,NY
1,Bean,32,Delhi
2,Charles,43,Mumbai
3,David,40,NaN


## read_sql()

**Purpose:** Load the result of a SQL query directly into a pandas DataFrame.

**Why it exists:** Most real-world company data lives in relational databases 
(PostgreSQL, MySQL, SQLite, SQL Server), not CSV files. ML/data work constantly 
requires pulling data straight from a database.

**Syntax:**
pd.read_sql(sql, con)

- sql: a SQL query string (or table name)
- con: a database connection object (created via SQLAlchemy)

**Return type:** DataFrame

**ML Usage:** Pulling training data, feature tables, or joined datasets directly 
from a company database instead of manually exporting CSVs.

**Important Notes:**
- Requires a connection engine, usually built with SQLAlchemy: 
  `from sqlalchemy import create_engine`
- Two sibling functions exist: `read_sql_query()` (query only) and 
  `read_sql_table()` (whole table, no query) — `read_sql()` handles both
- For learning, SQLite is the easiest — no server setup needed, just a local file

In [ ]:
import sqlite3
import pandas as pd

# create/connect to a local database file
conn = sqlite3.connect('examples/ex_database.db')

# create a table and insert data
conn.execute('''
CREATE TABLE IF NOT EXISTS people (
    name TEXT,
    age INTEGER,
    city TEXT
)
''')

conn.execute("DELETE FROM people")  # clear old data if rerunning
conn.executemany("INSERT INTO people VALUES (?, ?, ?)" , [('Leo', 25, 'Jaipur'), ('Binod', 30, 'Surat'), ('Charlie', 35, 'Pune')])
conn.commit()

In [60]:
df = pd.read_sql("SELECT * FROM people", conn)
df

# read_sql()
# Runs a SQL query against a connected database
# Returns the result directly as a DataFrame
# conn = the active database connection

,name,age,city
0,Leo,25,Jaipur
1,Binod,30,Surat
2,Charlie,35,Pune


In [61]:
conn.execute("INSERT INTO people VALUES (?, ?, ?)", ('David', 40, 'Bangalore'))
conn.commit()

pd.read_sql("SELECT name, city FROM people", conn)

,name,city
0,Leo,Jaipur
1,Binod,Surat
2,Charlie,Pune
3,David,Bangalore


## comment: strip trailing comments off lines
 `pd.read_csv('examples/ex1.csv', comment='#')`

## parse_dates: auto-convert a column to datetime
 `pd.read_csv('examples/ex1.csv', parse_dates=['some_date_col'])`

## converters: apply a function to every value in a column
 `pd.read_csv('examples/ex1.csv', converters={'message': str.upper})`

## thousands: interpret "1,000" as 1000, not a string
 `pd.read_csv('examples/ex1.csv', thousands=',')`

## encoding: needed for non-UTF8 files (common with Excel exports)
 `pd.read_csv('examples/ex1.csv', encoding='latin1')`

--> nrows/chunksize you already know from Step 7 

## "café" in UTF-8 → bytes: 63 61 66 C3 A9

## "café" in Latin-1 → bytes: 63 61 66 E9



In [62]:
import pandas as pd

# UTF-8 (default on most systems)
df = pd.read_csv("data.csv", encoding="utf-8")
# df

# Latin-1 (ISO-8859-1, common in legacy Windows files)
df = pd.read_csv("data.csv", encoding="latin1")

# Windows-1252 (similar to Latin-1, default on many Windows apps)
df = pd.read_csv("data.csv", encoding="cp1252")
